# TDSERAG — RAG agent con LangChain

Este notebook implementa desde cero los conceptos de RAG:

- **Indexing**: cargar → partir (chunking) → embeddings → vector store
- **Retrieval + Generation** (RAG):
  - Un **RAG chain** rápido (1 llamada al LLM por pregunta)
  - Un **RAG agent** con una tool de retrieval (y memoria opcional)

La configuración se lee desde el archivo [.env](.env).

## 0) Instalación

Ejecuta esta celda una sola vez por entorno/kernel.

In [ ]:
%pip install -q langchain langchain-google-genai langchain-core langchain-community langchain-text-splitters langchain-openai langchain-anthropic langgraph langchain-ollama python-dotenv bs4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1) Config (carga `.env` / `apikey.env`)

Este notebook lee configuración desde `.env`. Si existe un archivo `apikey.env`, también se carga (útil para guardar keys localmente).

Variables útiles:
- `LLM_PROVIDER`: `ollama` | `anthropic` | `openai` | `gemini`
- `LLM_MODEL`:
  - Ollama (local): usa exactamente lo que sale en `ollama list` (p.ej. `llama3:latest`)
  - Anthropic: p.ej. `claude-3-5-haiku-latest` / `claude-3-5-sonnet-latest`
  - Gemini: usa un modelo que aparezca en la celda de ListModels (ej. `models/gemini-2.0-flash`)
- Keys (según provider): `ANTHROPIC_API_KEY` / `OPENAI_API_KEY` / `GOOGLE_API_KEY`

Embeddings (Ollama):
- `OLLAMA_EMBED_MODEL`: p.ej. `nomic-embed-text` (recomendado)

Nota sobre agents + tools:
- El **RAG agent** usa tool calling. Algunos modelos de Ollama NO soportan tools; en ese caso el notebook hace fallback (retrieval explícito + 1 llamada al LLM).

Fuente a indexar:
- `SOURCE_URL`: URL del artículo (por defecto el de Lilian Weng)

Opcional (tracing):
- `LANGSMITH_TRACING=true`
- `LANGSMITH_API_KEY=...`

In [11]:
import os
from pathlib import Path

print("Working dir:", os.getcwd())
print("apikey.env exists:", Path("apikey.env").exists())

Working dir: c:\Users\Difga\Downloads\TDSERAG
apikey.env exists: True


In [12]:
from dotenv import dotenv_values
from pprint import pprint

config = dict(dotenv_values("apikey.env"))

def _redact(k: str, v: str | None):
    if v is None:
        return None
    if any(tok in k.upper() for tok in ["KEY", "TOKEN", "SECRET", "PASSWORD"]):
        return "***"
    return v

pprint({k: _redact(k, v) for k, v in config.items()})

{'CHUNK_OVERLAP': '200',
 'CHUNK_SIZE': '1000',
 'GOOGLE_API_KEY': '***',
 'LANGCHAIN_PROJECT': 'tdserag',
 'LANGSMITH_API_KEY': '***',
 'LANGSMITH_TRACING': 'false',
 'LLM_MAX_TOKENS': '***',
 'LLM_MODEL': 'llama3:latest',
 'LLM_PROVIDER': 'ollama',
 'LLM_TEMPERATURE': '0.2',
 'LLM_TIMEOUT': '60',
 'OLLAMA_EMBED_MODEL': 'nomic-embed-text',
 'SOURCE_URL': 'https://lilianweng.github.io/posts/2023-06-23-agent/',
 'TOP_K': '4'}


In [13]:
from __future__ import annotations

import os
from pathlib import Path
from dotenv import load_dotenv

# -----------------------------
# Cargar variables de entorno
# -----------------------------
load_dotenv(override=False)

if Path("apikey.env").exists():
    load_dotenv("apikey.env", override=True)

# -----------------------------
# LLM configuration
# -----------------------------
LLM_PROVIDER = (os.getenv("LLM_PROVIDER") or "ollama").strip().lower()

DEFAULT_MODELS = {
    "ollama": "llama3:latest",
    "anthropic": "claude-3-5-haiku-latest",
    "openai": "gpt-4o-mini",
    "gemini": "gemini-2.0-flash",
}

LLM_MODEL = (
    os.getenv("LLM_MODEL")
    or DEFAULT_MODELS.get(LLM_PROVIDER, "llama3:latest")
).strip()

# Normaliza nombres comunes que rompen en algunos SDKs/APIs
if LLM_PROVIDER == "gemini":
    # A veces viene como "models/gemini-2.0-flash" o "gemini-2.0-flash-latest"
    if LLM_MODEL.startswith("models/"):
        LLM_MODEL = LLM_MODEL.removeprefix("models/")
    if LLM_MODEL.endswith("-latest"):
        LLM_MODEL = LLM_MODEL.removesuffix("-latest")

LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE") or "0.2")
LLM_TIMEOUT = int(os.getenv("LLM_TIMEOUT") or "60")
LLM_MAX_TOKENS = int(os.getenv("LLM_MAX_TOKENS") or "1024")

# -----------------------------
# Embeddings (Ollama)
# -----------------------------
OLLAMA_EMBED_MODEL = (os.getenv("OLLAMA_EMBED_MODEL") or "nomic-embed-text").strip()

# -----------------------------
# Fuente a indexar (Web loader)
# -----------------------------
SOURCE_URL = (
    os.getenv("SOURCE_URL")
    or "https://lilianweng.github.io/posts/2023-06-23-agent/"
).strip()

# -----------------------------
# Validación de API keys
# -----------------------------
if LLM_PROVIDER == "anthropic" and not os.getenv("ANTHROPIC_API_KEY"):
    raise ValueError("ANTHROPIC_API_KEY no está configurada.")

if LLM_PROVIDER == "openai" and not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY no está configurada.")

if LLM_PROVIDER == "gemini" and not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("GOOGLE_API_KEY no está configurada.")

if LLM_PROVIDER not in {"ollama", "anthropic", "openai", "gemini"}:
    raise ValueError(f"Proveedor no soportado: {LLM_PROVIDER}")

print({
    "LLM_PROVIDER": LLM_PROVIDER,
    "LLM_MODEL": LLM_MODEL,
    "OLLAMA_EMBED_MODEL": OLLAMA_EMBED_MODEL,
    "SOURCE_URL": SOURCE_URL,
    "apikey.env_loaded": Path("apikey.env").exists(),
})

{'LLM_PROVIDER': 'ollama', 'LLM_MODEL': 'llama3:latest', 'OLLAMA_EMBED_MODEL': 'nomic-embed-text', 'SOURCE_URL': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'apikey.env_loaded': True}


In [14]:
from dotenv import dotenv_values
from pprint import pprint

config = dict(dotenv_values("apikey.env"))

def _redact(k: str, v: str | None):
    if v is None:
        return None
    if any(tok in k.upper() for tok in ["KEY", "TOKEN", "SECRET", "PASSWORD"]):
        return "***"
    return v

pprint({k: _redact(k, v) for k, v in config.items()})

{'CHUNK_OVERLAP': '200',
 'CHUNK_SIZE': '1000',
 'GOOGLE_API_KEY': '***',
 'LANGCHAIN_PROJECT': 'tdserag',
 'LANGSMITH_API_KEY': '***',
 'LANGSMITH_TRACING': 'false',
 'LLM_MAX_TOKENS': '***',
 'LLM_MODEL': 'llama3:latest',
 'LLM_PROVIDER': 'ollama',
 'LLM_TEMPERATURE': '0.2',
 'LLM_TIMEOUT': '60',
 'OLLAMA_EMBED_MODEL': 'nomic-embed-text',
 'SOURCE_URL': 'https://lilianweng.github.io/posts/2023-06-23-agent/',
 'TOP_K': '4'}


## 2) Inicializar modelo de chat + embeddings




In [16]:
import os

from langchain_ollama import OllamaEmbeddings

# --- Chat model ---
if LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI

    model = ChatGoogleGenerativeAI(
        model=LLM_MODEL,
        temperature=LLM_TEMPERATURE,
        google_api_key=os.getenv("GOOGLE_API_KEY"),
    )

elif LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI

    model = ChatOpenAI(
        model=LLM_MODEL,
        temperature=LLM_TEMPERATURE,
        timeout=LLM_TIMEOUT,
    )

elif LLM_PROVIDER == "anthropic":
    from langchain_anthropic import ChatAnthropic

    model = ChatAnthropic(
        model=LLM_MODEL,
        temperature=LLM_TEMPERATURE,
        timeout=LLM_TIMEOUT,
    )

elif LLM_PROVIDER == "ollama":
    from langchain_ollama import ChatOllama

    # Requiere que Ollama esté instalado y corriendo (http://localhost:11434).
    # Descarga un modelo de chat antes: ollama pull llama3
    model = ChatOllama(
        model=LLM_MODEL,
        temperature=LLM_TEMPERATURE,
    )

else:
    raise ValueError(f"Proveedor no soportado: {LLM_PROVIDER}")

# --- Embeddings (Ollama) ---
# Requiere que Ollama esté instalado y corriendo (http://localhost:11434).
embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL)

print({"chat_provider": LLM_PROVIDER, "chat_model": LLM_MODEL, "embed_model": OLLAMA_EMBED_MODEL})
(model, embeddings)

{'chat_provider': 'ollama', 'chat_model': 'llama3:latest', 'embed_model': 'nomic-embed-text'}


(ChatOllama(model='llama3:latest', temperature=0.2),
 OllamaEmbeddings(model='nomic-embed-text', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None))

In [17]:
# Preflight: valida que Ollama esté accesible y que el modelo de embeddings exista.
# Si esto falla, el vector store también fallará al indexar.
try:
    _ = embeddings.embed_query("ping")
    print("✅ Ollama embeddings OK")
except Exception as e:
    print("❌ No pude usar Ollama para embeddings.")
    print("Causado por:", repr(e))
    print("\nChecklist (Windows / PowerShell):")
    print("1) Instala Ollama: https://ollama.com/download")
    print("2) Asegúrate que corre el servicio (abre la app o ejecuta): ollama serve")
    print(f"3) Descarga el modelo: ollama pull {OLLAMA_EMBED_MODEL}")
    raise

✅ Ollama embeddings OK


## 3) Indexing: cargar documentos (Web loader)

Indexamos el post de Lilian Weng (igual que el tutorial).
Cambia `SOURCE_URL` en el `.env` 

In [18]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Mantener solo título, headers y contenido.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=(SOURCE_URL,),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

print(f"Loaded {len(docs)} document(s) from {SOURCE_URL}")
print(f"Total characters: {len(docs[0].page_content)}")
print(docs[0].page_content[:400])

USER_AGENT environment variable not set, consider setting it to identify your requests.


USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 document(s) from https://lilianweng.github.io/posts/2023-06-23-agent/
Total characters: 43047


      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, e


## 4) Indexing: split (chunking)

Usamos `RecursiveCharacterTextSplitter` con overlap.

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = int(os.getenv("CHUNK_SIZE") or "1000")
chunk_overlap = int(os.getenv("CHUNK_OVERLAP") or "200")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

print(f"Split into {len(all_splits)} chunks (chunk_size={chunk_size}, overlap={chunk_overlap}).")
all_splits[0]

Split into 63 chunks (chunk_size=1000, overlap=200).


Split into 63 chunks (chunk_size=1000, overlap=200).


Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 8}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from 

## 5) Indexing: vector store (In-memory)

Como en el tutorial, usamos `InMemoryVectorStore`.

In [20]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
doc_ids = vector_store.add_documents(documents=all_splits)

print(f"Indexed {len(doc_ids)} chunks")
doc_ids[:3]

Indexed 63 chunks


['fd86c902-8a75-467f-b12d-e61391fc9d14',
 '1610829a-3446-4a8c-b02e-1ee8ffac4f39',
 'fadbece1-b3ca-4746-842f-2502ed53acda']

## 6) Retrieval + Generation: RAG chain (1 llamada al LLM)

Patrón:
1) Retrieve top-k chunks
2) Una sola llamada al LLM con pregunta + contexto

In [21]:
print({
    "LLM_PROVIDER": LLM_PROVIDER,
    "LLM_MODEL": LLM_MODEL,
})

{'LLM_PROVIDER': 'ollama', 'LLM_MODEL': 'llama3:latest'}


In [28]:
from typing import List

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

TOP_K = int(os.getenv("TOP_K") or "4")
retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

def format_docs(docs: List[Document]) -> str:
    lines = []
    for i, d in enumerate(docs, start=1):
        source = d.metadata.get("source", "unknown")
        start = d.metadata.get("start_index", "?")
        lines.append(f"[Chunk {i}] source={source} start_index={start}\n{d.page_content}")
    return "\n\n".join(lines)

RAG_SYSTEM = (
    "You are a Q&A assistant. Answer the user ONLY using the provided context. "
    "If the context is insufficient, say you don't know."
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM),
    ("human", "Question: {question}\n\nContext:\n{context}"),
])

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

question = "whats is a agent system overview?"
answer = rag_chain.invoke(question)
print(answer)

Based on the provided context, an Agent System Overview is a framework that describes how Large Language Models (LLMs) can be used as the core controller of autonomous agents. The overview highlights two key components:

1. Planning:
	* Subgoal and decomposition: breaking down large tasks into smaller, manageable subgoals.
	* Reflection and refinement: self-criticism, learning from mistakes, and refining actions for future steps.

2. Memory:
	* Not explicitly described in the provided context, but it is mentioned as one of the key components in the generative agent architecture image (Chunk 4).

The overview also mentions that LLMs can be used to create autonomous agents with capabilities such as information diffusion, relationship memory, and coordination of social events.


In [91]:
docs = retriever.invoke("What says the Scientific Discovery Agent?")
print(format_docs(docs))

[Chunk 1] source=https://lilianweng.github.io/posts/2023-06-23-agent/ start_index=32858
}
]
Then after these clarification, the agent moved into the code writing mode with a different system message.
System message:

[Chunk 2] source=https://lilianweng.github.io/posts/2023-06-23-agent/ start_index=21014
Pseudo code of how LLM makes an API call in API-Bank. (Image source: Li et al. 2023)

In the API-Bank workflow, LLMs need to make a couple of decisions and at each step we can evaluate how accurate that decision is. Decisions include:

Whether an API call is needed.
Identify the right API to call: if not good enough, LLMs need to iteratively modify the API inputs (e.g. deciding search keywords for Search Engine API).
Response based on the API results: the model can choose to refine and call again if results are not satisfied.

This benchmark evaluates the agent’s tool use capabilities at three levels:

[Chunk 3] source=https://lilianweng.github.io/posts/2023-06-23-agent/ start_index=301

## 7) RAG agent: retrieval tool + memoria (quickstart)

El agent decide cuándo llamar `retrieve_context`. Además usamos memoria con `InMemorySaver` y `thread_id`.

Esta sección incluye fallbacks por si tu versión instalada de LangChain expone una API distinta.

In [24]:
import os
from dataclasses import dataclass

from langchain.tools import tool
from langchain_core.documents import Document
from langgraph.checkpoint.memory import InMemorySaver

# Asegura TOP_K incluso si no ejecutaste la celda del RAG chain todavía.
TOP_K = int(os.getenv("TOP_K") or "4")

checkpointer = InMemorySaver()

@dataclass
class Context:
    """Runtime context schema (opcional)."""
    user_id: str = "default"
    top_k: int = TOP_K

def _retrieve_context(query: str, k: int) -> list[Document]:
    return vector_store.similarity_search(query, k=k)

def _serialize_docs(retrieved_docs: list[Document]) -> str:
    return "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )

def _tool_impl(query: str) -> tuple[str, list[Document]]:
    retrieved_docs = _retrieve_context(query=query, k=TOP_K)
    serialized = _serialize_docs(retrieved_docs)
    return serialized, retrieved_docs

try:
    @tool(response_format="content_and_artifact")
    def retrieve_context(query: str):
        """Retrieve information to help answer a query."""
        return _tool_impl(query)
except TypeError:
    # Compatibilidad con versiones donde response_format no exista.
    @tool
    def retrieve_context(query: str) -> str:
        """Retrieve information to help answer a query."""
        serialized, _docs = _tool_impl(query)
        return serialized

tools = [retrieve_context]

SYSTEM_PROMPT = (
    "You are a helpful RAG assistant. You have access to a retrieval tool. "
    "Use the tool to look up relevant context before answering. "
    "If you cannot find the answer in the retrieved context, say you don't know."
 )

@dataclass
class ResponseFormat:
    """Schema de respuesta del agent."""
    answer: str
    sources: list[str] | None = None

response_format = None
try:
    from langchain.agents.structured_output import ToolStrategy
    response_format = ToolStrategy(ResponseFormat)
except Exception:
    response_format = None

agent = None
agent_mode = None
try:
    from langchain.agents import create_agent

    agent = create_agent(
        model=model,
        tools=tools,
        system_prompt=SYSTEM_PROMPT,
        context_schema=Context,
        response_format=response_format,
        checkpointer=checkpointer,
    )
    agent_mode = "create_agent"
except Exception as e:
    print("Falling back to AgentExecutor API (create_agent not available):", repr(e))
    from langchain.agents import AgentExecutor, create_tool_calling_agent
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ])
    tool_calling_agent = create_tool_calling_agent(model, tools, prompt)
    agent = AgentExecutor(agent=tool_calling_agent, tools=tools, verbose=True)
    agent_mode = "agent_executor"

print({"agent_mode": agent_mode, "TOP_K": TOP_K})

{'agent_mode': 'create_agent', 'TOP_K': 4}


### Qué hace esta parte (ejecución del Agent / fallback)

En la siguiente celda se **ejecuta una pregunta** contra el sistema RAG que ya construiste (modelo + embeddings + `vector_store`).

**Flujo general:**
1) Defines `query` (tu pregunta).
2) Valida que existan `vector_store` y `model` (si no, te pide correr las secciones anteriores).
3) Intenta usar el **agent** (si fue inicializado en la celda anterior).
4) Si el agent no existe, o si tu modelo no soporta *tool calling*, se usa un **fallback**:
   - retrieval explícito (`vector_store.similarity_search`)
   - 1 llamada al LLM con el contexto recuperado

**Qué puedes cambiar rápido:**
- Cambia el string de `query` para preguntar otra cosa del artículo.
- Ajusta `TOP_K` en `apikey.env` si quieres traer más/menos chunks.

**Nota:** La idea del notebook es que las respuestas estén **ancladas al texto indexado** (el post de Lilian Weng). Si preguntas algo que no está en ese contenido, lo correcto es que el sistema responda que no hay suficiente contexto o que está fuera de alcance.

In [27]:
import os

query = (
    "whats is a agent system overview?"
 )

# --------- Robust config (no depende de que hayas corrido todas las celdas) ---------
_provider = globals().get("LLM_PROVIDER") or (os.getenv("LLM_PROVIDER") or "ollama").strip().lower()
_top_k = int(globals().get("TOP_K") or (os.getenv("TOP_K") or "4"))

if "vector_store" not in globals():
    raise RuntimeError(
        "No encuentro `vector_store` en memoria. Corre primero las celdas de Indexing (secciones 3–5) "
        "para construir el vector store y luego vuelve a correr esta celda."
    )

if "model" not in globals():
    raise RuntimeError(
        "No encuentro `model` en memoria. Corre primero la celda de inicialización del modelo (sección 2) "
        "y luego vuelve a correr esta celda."
    )

_SYSTEM_PROMPT = globals().get("SYSTEM_PROMPT") or (
    "You are a helpful RAG assistant. Use the provided context to answer. "
    "If the context is insufficient, say you don't know."
 )

def _serialize_docs_local(retrieved_docs):
    # Intenta usar el serializer existente si está definido en el notebook
    if "_serialize_docs" in globals():
        return globals()["_serialize_docs"](retrieved_docs)
    return "\n\n".join(
        f"Source: {getattr(doc, 'metadata', {})}\nContent: {getattr(doc, 'page_content', str(doc))}"
        for doc in retrieved_docs
    )

def _fallback_answer_with_retrieval(user_query: str) -> str:
    """Fallback cuando no hay agent o cuando el modelo no soporta tool calling."""
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate

    retrieved_docs = vector_store.similarity_search(user_query, k=_top_k)
    context = _serialize_docs_local(retrieved_docs)

    prompt = ChatPromptTemplate.from_messages([
        ("system", _SYSTEM_PROMPT + "\n\nIf you use the context, cite it briefly."),
        ("human", "Question:\n{question}\n\nRetrieved context:\n{context}"),
    ])

    chain = prompt | model | StrOutputParser()
    return chain.invoke({"question": user_query, "context": context})

# --------- Run agent if available; otherwise fallback ---------
_agent_mode = globals().get("agent_mode")  # puede no existir si no corriste la celda anterior
_agent = globals().get("agent")

try:
    if _agent is None or _agent_mode is None:
        print("[Info] Agent no inicializado; usando fallback (retrieval explícito + 1 llamada al LLM).")
        print(_fallback_answer_with_retrieval(query))
    elif _agent_mode == "create_agent":
        config = {"configurable": {"thread_id": "demo-thread-1"}}
        response = _agent.invoke(
            {"messages": [{"role": "user", "content": query}]},
            config=config,
            context=globals().get("Context", object)(),
        )
        if isinstance(response, dict) and response.get("structured_response") is not None:
            print(response["structured_response"])
        else:
            messages = response.get("messages", []) if isinstance(response, dict) else []
            print(messages[-1].content if messages else response)
    else:
        response = _agent.invoke({"input": query})
        print(response.get("output", response))
except Exception as e:
    msg = str(e).lower()
    # Caso típico: Ollama chat model sin soporte de tools en modo agent
    if _provider == "ollama" and "does not support tools" in msg:
        print("[Fallback] Este modelo de Ollama no soporta tool calling; usando retrieval explícito + 1 llamada al LLM.")
        print(_fallback_answer_with_retrieval(query))
    else:
        raise

[Fallback] Este modelo de Ollama no soporta tool calling; usando retrieval explícito + 1 llamada al LLM.
Based on the retrieved context, an Agent System Overview refers to a system that uses Large Language Models (LLMs) as its core controller, complemented by key components such as Planning and Memory. The planning component enables the agent to break down large tasks into smaller subgoals, reflect on past actions, and refine them for future steps. The memory component allows the agent to store and retrieve information, enabling emergent social behavior such as information diffusion and coordination of social events.

Citation: Weng, Lilian. (Jun 2023). “LLM-powered Autonomous Agents”. Lil’Log. https://lilianweng.github.io/posts/2023-06-23-agent/.
